In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device

In [ ]:
state_dict_student = torch.load("student_model_99_0.8380.pth")
type(state_dict_student)

In [ ]:
from llama3_2.config import LlamaConfig
from llama3_2.model_trfmrs import LlamaModel
import safetensors.torch as st

vocab_size = 32768
student_config = LlamaConfig(vocab_size=vocab_size)
student_model1 = LlamaModel(student_config)
student_model1 = student_model1.to(device)

student_model_states = st.load_file("student_model1.safetensors")
student_model1.load_state_dict(state_dict_student)

student_model2 = LlamaModel(student_config)
student_model2 = student_model2.to(device)

student_model2.load_state_dict(student_model_states)

In [ ]:
v1 = student_model1(torch.tensor([[64]]).to(device))
v1[0, 0, -6:]

In [ ]:
v2 = student_model2(torch.tensor([[64]]).to(device))
v2[0, 0, -6:]

In [ ]:
teacher_config = LlamaConfig()
teacher_model = LlamaModel(teacher_config)
teacher_model = teacher_model.to(device)

teacher_model_states = st.load_file("teacher_model1.safetensors")
teacher_model.load_state_dict(teacher_model_states)

In [ ]:
v2 = student_model2(torch.tensor([[64]]).to(device))
v2

In [ ]:
vt = teacher_model(torch.tensor([[64]]).to(device))
vt

In [ ]:
new_token_tensor = torch.tensor([[64]]).to(device)

teacher_model_output = teacher_model(new_token_tensor)
student_model_out = student_model2(new_token_tensor)

print(teacher_model_output.shape, student_model_out.shape, torch.mean(teacher_model_output, dim=1), torch.mean(student_model_out, dim=1))


In [ ]:
import torch

def mean_squared_error(y_true, y_pred):
  return torch.mean((y_true - y_pred) ** 2)


mse = mean_squared_error(v2, vt)
mse


In [16]:
same_embeddings = []

In [ ]:
for i in range(32768):
  v2 = student_model2(torch.tensor([[i]]).to(device))
  vt = teacher_model(torch.tensor([[i]]).to(device))
  mse = mean_squared_error(v2, vt)
  if i % 100 == 0:
    print(i, mse)
  if mse < 0.0001:
    same_embeddings.append(i)
    print(i, mse)

len(same_embeddings)